# A Textbook-Style Exposition of Hybrid ALNS for the One-Dimensional Bin Packing Problem

## I. Formal Problem Statement

### I.1 Optimization problem and input model
The one-dimensional bin packing problem is defined as follows. We are given a finite set of items indexed by $\{1,2,\ldots,n\}$, with strictly positive integer sizes $s_i \in \mathbb{Z}_{>0}$, and identical bins of capacity $C \in \mathbb{Z}_{>0}$.

A feasible solution is a partition of the item index set into subsets $B_1, B_2, \ldots, B_m$ such that:
- $\biguplus_{j=1}^{m} B_j = \{1,2,\ldots,n\}$,
- $\sum_{i \in B_j} s_i \le C$ for every $j \in \{1,2,\ldots,m\}$.

The objective is to minimize the number of bins:
$$
\min \; m.
$$

### I.2 Lower bound and computational complexity perspective
A canonical lower bound is
$$
\mathrm{LB} = \left\lceil \frac{\sum_{i=1}^{n} s_i}{C} \right\rceil.
$$
This bound is valid because total packed volume cannot exceed total available capacity.

From the standpoint of classical combinatorial optimization, the problem is NP-hard. Consequently, for large instances one often deploys approximation algorithms, constructive heuristics, and metaheuristics, rather than relying exclusively on exact optimization.

---

## II. Metaheuristic Framework: From Large-Neighborhood Search to Adaptive Large-Neighborhood Search

### II.1 Motivation for large-neighborhood mechanisms
In local search, small neighborhoods can induce severe entrapment in local minima. Large-neighborhood search (LNS) addresses this limitation by repeatedly performing:
1. a **destroy** operation that partially unassigns items, and
2. a **repair** operation that reconstructs feasibility.

Because reconstruction can alter many assignments simultaneously, LNS can traverse regions of the solution space inaccessible to small move operators.

### II.2 Adaptive Large-Neighborhood Search (ALNS)
ALNS augments LNS by maintaining multiple operators and adapting their usage frequencies as evidence accumulates during the run.

In the present solver, the destroy phase employs several operators (for example, random destruction, destruction of poorly loaded structures, and related-item destruction), while the repair phase is learned. Operator selection is controlled adaptively through a bandit mechanism.

---

## III. Acceptance Mechanism: Simulated Annealing Principle

Let $x$ denote the incumbent solution, $x'$ a candidate produced by destroy-repair, and $f(\cdot)$ the objective (number of used bins). Define
$$
\Delta = f(x') - f(x).
$$

The acceptance rule is:
- if $\Delta \le 0$, accept deterministically;
- if $\Delta > 0$, accept with probability
$$
\mathbb{P}(\text{accept } x' \mid x) = \exp\!\left(-\frac{\Delta}{T}\right),
$$
where $T>0$ is the temperature parameter.

With a geometric cooling law, such as $T \leftarrow \alpha T$ for $\alpha \in (0,1)$, the search gradually transitions from exploratory to exploitative behavior, which is a classical diversification-intensification compromise.

---

## IV. Adaptive Operator Control via Thompson Sampling

### IV.1 Bandit model
Each destroy operator is interpreted as an arm in a stochastic multi-armed bandit model. For arm $k$, let an unknown Bernoulli success parameter be $\theta_k$.

A conjugate Bayesian model is maintained:
$$
\theta_k \sim \mathrm{Beta}(\alpha_k, \beta_k).
$$

### IV.2 Decision and update cycle
At each iteration:
1. sample $\tilde{\theta}_k \sim \mathrm{Beta}(\alpha_k,\beta_k)$ for each arm $k$,
2. select the arm $k^* = \arg\max_k \tilde{\theta}_k$,
3. observe a binary reward (for example, whether an accepted improving move was generated),
4. update $(\alpha_{k^*},\beta_{k^*})$ accordingly.

This procedure yields a principled exploration-exploitation trade-off and is consistent with standard Bayesian bandit methodology.

---

## V. Machine-Learned Repair Component

### V.1 Repair as a structured decision subproblem
After destruction, each displaced item must be reinserted. For a fixed item, several bins may be feasible. The repair choice can be posed as a supervised scoring problem over feasible $(\text{item}, \text{bin})$ pairs.

### V.2 Feature representation and inference
For each feasible pair, one computes feature vectors that encode, for example, item magnitude, current bin load, residual capacity, and post-placement slack. A logistic model outputs a score/probability; the feasible bin with the highest score is selected. If no feasible bin exists, a new bin is opened.

### V.3 Offline training protocol
Training data is generated by replaying a strong constructive reference rule (Best-Fit Decreasing) and labeling:
- positive examples as the feasible bin selected by the reference rule,
- negative examples as alternative feasible bins.

Thus, the model learns a data-driven approximation to constructive repair preferences, while ALNS provides higher-level global search dynamics.

---

## VI. Minimal Reproducible Workflow in This Repository

The practical sequence is intentionally minimal:
1. train the repair model and serialize it as $\texttt{repair\_model.pkl}$,
2. invoke the benchmark driver with the hybrid solver,
3. pass the serialized model path through method arguments,
4. inspect the textual benchmark summary.

The two code cells below implement precisely this workflow and intentionally avoid plotting, so that execution remains simple and reproducible.


In [ ]:
!python train_repair_model.py \
  --instances 5000 \
  --n-min 50 \
  --n-max 200 \
  --max-negatives 5 \
  --seed 0 \
  --output repair_model.pkl

In [ ]:
!python ../../benchmark.py \
  --solver 5_hybrid_ml_metaheuristics/hybrid_alns/solver.py \
  --method hybrid_alns \
  --method-args model_path=5_hybrid_ml_metaheuristics/hybrid_alns/repair_model.pkl,max_iterations=5000 \
  --datasets falkenauer-t \
  --limit 5